# Phase VII Work Report: Network Topology and Bipartite Graph Construction

in this phase I tried to construct a comprehensive bipartite network graph mapping the interactions between individual wallets (traders) and smart contracts (liquidity pools). By transforming a linear transaction ledger into a mathematical network, I aimed to quantify trader specialization, measure ecosystem concentration, and isolate Maximum Extractable Value (MEV) actors based on their structural routing behavior.

## Methodology

1. **Dynamic Schema and Temporal Projection:**  
   I initiated the pipeline by building a dynamic schema resolution engine. This algorithm automatically standardizes column identifiers regardless of upstream variations, mapping wallets, pools, and timestamps definitively. To optimize memory consumption across tens of millions of rows, I applied a narrow temporal projection, extracting only the essential calendar indices required for calculating network persistence.

2. **Structural MEV Signal Extraction:**  
   Rather than relying solely on the previous event-level classifications, I aggregated the data to the parent transaction hash level. I mathematically defined a true cyclic arbitrageur as any entity executing a multi-leg transaction where the input token perfectly matches the output token. Similarly, I mapped sandwich attack slots by identifying wallets that executed multiple distinct transactions against the exact same pool within the exact same block. I established strict, base, and loose econometric thresholds based on the statistical quantiles of these behaviors.

3. **Bucketed Edge Generation:**  
   Because generating a full bipartite edge list across sixty million swaps exceeds standard RAM capacities, I engineered a deterministic hash-bucketing algorithm. By partitioning the wallets into sixteen distinct memory buckets, I safely aggregated the pairwise edge metrics—calculating exact trading volumes, temporal persistence (span and active months), and return rates between specific wallets and pools.

4. **Topological Node Metrics:**  
   Utilizing the bucketed edges, I computed advanced network topology metrics for both the wallet and pool nodes. I calculated the Herfindahl-Hirschman Index (HHI) and normalized Shannon Entropy to mathematically represent concentration versus diversification. A low entropy, high HHI wallet represents a highly specialized actor (trading in one specific pool), whereas high entropy denotes an exploratory or algorithmic routing entity. Finally, I derived a composite MEV Exposure Index for every liquidity pool by ranking their proportional interaction with identified algorithmic actors.

In [1]:
from config import OUT
import os, math
import polars as pl
from pathlib import Path
import re
import gc, shutil

print(f"Saving data to: {OUT}")

Saving data to: C:\Users\Pouyan\python\thesis\Proposal\FINAL\Thesis_Output


---
## Output Configuration and Column Mapping
This cell initializes the output directory and establishes a standardized dictionary to map the columns from the upstream dataset into the strict naming convention required for the network topology model. It also applies a lazy frame projection, parsing the timestamps into exact months and days to facilitate temporal network analysis later in the pipeline.

In [5]:
OUT = Path("./Thesis_Output")
A = OUT / "Dataset_A_Final.parquet"          
G = OUT / "Dataset_G_TraderHistory.parquet"  

# Map upstream Dataset A column names to standardized network variables
COLMAP = {
    "wallet":     "trader",        
    "pool":       "pool_address",  
    "ts":         "block_timestamp",
    "block":      "block_number",
    "volume_usd": "amount_usd",
}

lf = pl.scan_parquet(A).rename({v: k for k, v in COLMAP.items()})

lf = lf.with_columns(
    pl.col("volume_usd").cast(pl.Float64).fill_null(0.0),
    (pl.col("ts").dt.year() * 12 + pl.col("ts").dt.month()).alias("month_idx"),
    pl.col("ts").dt.date().alias("day"),
)

---
## Dynamic Schema Resolution
Because upstream dataset column names frequently change during data engineering iteration, this cell implements a dynamic resolution algorithm. It scans the available schema against a dictionary of highly likely candidate names, utilizing regular expressions to strip punctuation and casing. If the algorithm cannot definitively locate a required column like the wallet address or block number, it explicitly halts the pipeline, preventing silent failures.

In [10]:
# Define candidate column names ranked by probability of occurrence
CANDIDATES = {
    "wallet": ["trader","wallet","wallet_address","trader_address","sender","sender_address",
               "from_address","tx_from","origin_from_address","origin","eoa","user","user_address",
               "account","maker","taker","swapper","tx_sender","recipient","to_address"],
    "pool":   ["pool","pool_address","pool_id","poolid","pair","pair_address","market",
               "liquidity_pool","pool_contract","contract_address"],
    "block":  ["block","block_number","blocknumber","block_num","block_height","height","blk"],
    "ts":     ["ts","timestamp","block_timestamp","block_time","blocktime","evt_block_time",
               "block_signed_at","time","datetime","date_time","date"],
    "volume_usd": ["volume_usd","amount_usd","amountusd","usd_value","value_usd","usd_amount",
                   "notional_usd","trade_size_usd","size_usd","amount_usd_abs","usd"],
}

# Provide manual overrides if the dynamic resolution guesses incorrectly
MANUAL = {}

def _norm(s):  
    # Create a case and punctuation insensitive key for matching
    return re.sub(r"[^a-z0-9]", "", s.lower())

# Explicitly extract the schema from the base file before executing the matching logic
schema = pl.scan_parquet(A).collect_schema()

_avail = {_norm(c): c for c in schema.names()}

src, missing = {}, []
for canon, cands in CANDIDATES.items():
    if canon in MANUAL:
        src[canon] = MANUAL[canon]
        continue
    hit = next((_avail[_norm(c)] for c in cands if _norm(c) in _avail), None)
    
    if hit is None:                                  
        # Fallback to a loose substring matching pass
        hit = next((orig for k, orig in _avail.items()
                    if any(_norm(c) in k for c in cands)), None)
    if hit: 
        src[canon] = hit
    else:   
        missing.append(canon)

print("RESOLVED COLUMN MAP")
for k in CANDIDATES:
    print(f"  {k:<12} mapped to  {src.get(k, 'NOT FOUND')}")

# Enforce strict existence checks for mathematically mandatory columns
hard = [c for c in ("wallet","pool","block") if c not in src]
if hard:
    raise KeyError(
        f"Cannot proceed no column found for {hard}. "
        f"Available columns are {schema.names()}. "
        f"Fix by setting MANUAL dictionary."
    )
if missing:
    print(f"\nSoft missing columns will be mathematically synthesized: {missing}")

RESOLVED COLUMN MAP
  wallet       mapped to  wallet
  pool         mapped to  pool_address
  block        mapped to  block_number
  ts           mapped to  block_time
  volume_usd   mapped to  amount_usd


---
## Temporal Inference and Lazy Projection
This cell sanitizes the resolved columns and infers the correct temporal format. If standard timestamps are unavailable, it synthesizes pseudo-time using block heights to ensure network edge persistence calculations do not fail. Crucially, it then aggressively narrows the dataframe projection down to just seven columns, resulting in massive computational speedups when processing millions of swaps.

In [13]:
lf = pl.scan_parquet(A).rename({v: k for k, v in src.items()})

# Infer timestamp storage format to prevent silent epoch truncation errors
if "ts" in src:
    dt = schema[src["ts"]]
    if dt in (pl.Datetime, pl.Datetime("us"), pl.Datetime("ms"), pl.Datetime("ns")) or str(dt).startswith("Datetime"):
        pass
    elif dt == pl.Date:
        lf = lf.with_columns(pl.col("ts").cast(pl.Datetime("us")))
    elif str(dt) in ("String", "Utf8", "LargeUtf8"):
        lf = lf.with_columns(pl.col("ts").str.to_datetime(strict=False))
    else:  
        # Detect integer epoch unit from total magnitude
        mx = pl.scan_parquet(A).select(pl.col(src["ts"]).max()).collect().item()
        unit = ("s" if mx < 1e11 else "ms" if mx < 1e14 else "us" if mx < 1e17 else "ns")
        print(f"Epoch detected with maximum {mx} mapping to unit {unit}")
        lf = lf.with_columns(pl.from_epoch(pl.col("ts").cast(pl.Int64), time_unit=unit).alias("ts"))
else:
    # Synthesize a pseudo time index based on the average Ethereum block time of 12 seconds
    print("No timestamp column detected synthesizing pseudo time from block height")
    lf = lf.with_columns(
        (pl.lit(pl.datetime(2020,1,1)) +
         pl.duration(seconds=(pl.col("block").cast(pl.Int64) -
                              pl.col("block").cast(pl.Int64).min()) * 12)).alias("ts")
    )

if "volume_usd" in src:
    lf = lf.with_columns(pl.col("volume_usd").cast(pl.Float64).fill_null(0.0).abs())
else:
    print("No USD column detected volume metrics will fall back to raw swap counts")
    lf = lf.with_columns(pl.lit(1.0).alias("volume_usd"))

# Apply calendar indices and aggressive column projection to maximize execution speed
lf = (lf.with_columns([
          (pl.col("ts").dt.year() * 12 + pl.col("ts").dt.month()).alias("month_idx"),
          pl.col("ts").dt.date().alias("day"),
          pl.col("block").cast(pl.Int64),
      ])
      .select(["wallet","pool","block","ts","volume_usd","month_idx","day"]))

print("Projection initialization successful schema validated")
print(lf.collect_schema())
print(lf.head(5).collect())

Projection initialization successful schema validated
Schema({'wallet': String, 'pool': String, 'block': Int64, 'ts': Datetime(time_unit='us', time_zone=None), 'volume_usd': Float64, 'month_idx': Int32, 'day': Date})
shape: (5, 7)
┌───────────────┬───────────────┬──────────┬───────────────┬──────────────┬───────────┬────────────┐
│ wallet        ┆ pool          ┆ block    ┆ ts            ┆ volume_usd   ┆ month_idx ┆ day        │
│ ---           ┆ ---           ┆ ---      ┆ ---           ┆ ---          ┆ ---       ┆ ---        │
│ str           ┆ str           ┆ i64      ┆ datetime[μs]  ┆ f64          ┆ i32       ┆ date       │
╞═══════════════╪═══════════════╪══════════╪═══════════════╪══════════════╪═══════════╪════════════╡
│ 0x93793bd1f3e ┆ 0x531b6a4b3f9 ┆ 21525891 ┆ 2025-01-01    ┆ 66584.150285 ┆ 24301     ┆ 2025-01-01 │
│ 35a0efd098c30 ┆ 62208ea8ed526 ┆          ┆ 00:00:11      ┆              ┆           ┆            │
│ e486…         ┆ 8c64…         ┆          ┆               ┆  

---
## Transaction Level MEV Signal Extraction
This cell reconstructs full transactions from individual swap legs. It mathematically identifies true cyclic arbitrage (a multi-hop route where the initial asset precisely matches the final asset) and isolates sandwich attack vulnerabilities by finding wallets executing multiple distinct transactions against a single pool within the exact same block. It then aggregates these signals to the wallet level to establish behavioral profiles.

In [16]:
print("Extracting transaction level MEV routing signals")

BLK_CACHE = OUT / "_H_tx_cache.parquet"

# Re-scan the dataset retaining leg level routing columns dropped in the previous projection
lf_tx = (pl.scan_parquet(A)
           .select([
               pl.col(src["wallet"]).alias("wallet"),
               pl.col("tx_hash"),
               pl.col("log_index").cast(pl.Int64),
               pl.col(src["pool"]).alias("pool"),
               pl.col(src["block"]).cast(pl.Int64).alias("block"),
               pl.col("token_in"), pl.col("token_out"),
               pl.col(src["volume_usd"]).cast(pl.Float64).fill_null(0.0).abs().alias("volume_usd"),
           ]))

# Reconstruct transactions to analyze multileg routing behavior
tx = (lf_tx.group_by(["wallet", "tx_hash", "block"])
        .agg([
            pl.len().alias("n_legs"),
            pl.col("pool").n_unique().alias("n_pools"),
            pl.col("token_in").sort_by("log_index").first().alias("tok_first_in"),
            pl.col("token_out").sort_by("log_index").last().alias("tok_last_out"),
            pl.col("token_in").n_unique().alias("n_tok_in"),
            pl.col("volume_usd").max().alias("tx_usd"),
        ])
        .with_columns([
            # Mathematically define true cyclic arbitrage closing the token loop
            ((pl.col("tok_first_in") == pl.col("tok_last_out")) &
             (pl.col("n_legs") >= 2)).alias("is_cycle"),
            
            # Identify split routing aggregator usage separate from true cycles
            ((pl.col("n_pools") >= 2) &
             (pl.col("tok_first_in") != pl.col("tok_last_out"))).alias("is_split_route"),
            (pl.col("n_legs") >= 3).alias("is_multihop"),
        ]))

try:
    tx.sink_parquet(BLK_CACHE, compression="zstd")
except Exception as e:
    print(f"Streaming sink failed triggering memory fallback {type(e).__name__}")
    tx.collect(engine="streaming").write_parquet(BLK_CACHE, compression="zstd")

tx_lf = pl.scan_parquet(BLK_CACHE)
print(f"Total parent transactions analyzed {tx_lf.select(pl.len()).collect().item():,}")

# Isolate sandwich signals identical wallet and pool but distinct transaction hashes within one block
sandwich = (lf_tx.group_by(["wallet", "block", "pool"])
              .agg(pl.col("tx_hash").n_unique().alias("n_tx_same_pool_block"))
              .filter(pl.col("n_tx_same_pool_block") >= 2)
              .group_by("wallet")
              .agg([pl.len().alias("n_sandwich_slots"),
                    pl.col("n_tx_same_pool_block").max().alias("max_tx_same_pool_block")])
              .collect(engine="streaming"))

# Profile the wallets based on their aggregated transaction routing signals
wallet_mev = (
    tx_lf.group_by("wallet")
      .agg([
          pl.len().alias("n_tx"),
          pl.col("is_cycle").mean().alias("arb_tx_rate"),
          pl.col("is_cycle").sum().alias("n_arb_tx"),
          pl.col("is_split_route").mean().alias("split_route_rate"),
          pl.col("is_multihop").mean().alias("multihop_rate"),
          pl.col("n_legs").max().alias("max_legs_one_tx"),
          pl.col("block").n_unique().alias("n_active_blocks"),
      ])
      .collect(engine="streaming")
      .join(sandwich, on="wallet", how="left")
      .with_columns([pl.col("n_sandwich_slots").fill_null(0),
                     pl.col("max_tx_same_pool_block").fill_null(0)])
      .with_columns([
          (pl.col("n_tx") / pl.col("n_active_blocks")).alias("tx_per_active_block"),
      ])
      .with_columns(
          # Define conservative heuristics requiring repeated behavioral patterns to classify as MEV
          (((pl.col("arb_tx_rate") > 0.50) & (pl.col("n_arb_tx") >= 5)) |
           (pl.col("n_arb_tx") >= 50) |
           (pl.col("n_sandwich_slots") >= 10)).alias("mev_wallet")
      )
)

n, k = wallet_mev.height, int(wallet_mev["mev_wallet"].sum())
print(f"Total wallets {n:,} Classified MEV actors {k:,} representing {k/n:.2%}")
print(wallet_mev.select(["arb_tx_rate","split_route_rate","n_arb_tx",
                        "n_sandwich_slots","max_legs_one_tx"]).describe())

Extracting transaction level MEV routing signals
Total parent transactions analyzed 45,532,663
Total wallets 3,987,674 Classified MEV actors 2,130 representing 0.05%
shape: (9, 6)
┌────────────┬─────────────┬──────────────────┬────────────┬──────────────────┬─────────────────┐
│ statistic  ┆ arb_tx_rate ┆ split_route_rate ┆ n_arb_tx   ┆ n_sandwich_slots ┆ max_legs_one_tx │
│ ---        ┆ ---         ┆ ---              ┆ ---        ┆ ---              ┆ ---             │
│ str        ┆ f64         ┆ f64              ┆ f64        ┆ f64              ┆ f64             │
╞════════════╪═════════════╪══════════════════╪════════════╪══════════════════╪═════════════════╡
│ count      ┆ 3.987674e6  ┆ 3.987674e6       ┆ 3.987674e6 ┆ 3.987674e6       ┆ 3.987674e6      │
│ null_count ┆ 0.0         ┆ 0.0              ┆ 0.0        ┆ 0.0              ┆ 0.0             │
│ mean       ┆ 0.001278    ┆ 0.15748          ┆ 0.253179   ┆ 0.136432         ┆ 1.394342        │
│ std        ┆ 0.031839    ┆ 0.31148

---
## Thread Optimization for Hash Operations
Polars inherently maximizes all available CPU cores. However, during massive out-of-core grouped hash operations, too many threads cause lock contention and severe RAM spiking. Lowering the maximum threads stabilizes the memory pressure before executing the bipartite edge joins.

In [19]:
# Constrain thread count to mitigate memory spiking and lock contention during heavy hash aggregations
os.environ["POLARS_MAX_THREADS"] = "4"

---
## Empirical MEV Threshold Calibration
This cell performs an empirical calibration of MEV actors. By printing the statistical quantiles of cyclic behavior, it proves that arbitrage is highly concentrated among a select few algorithms. It then defines three nested classifications (strict, base, loose) to allow for sensitivity analysis in the final thesis models, exporting the classifications to disk.

In [22]:
print("Calibrating structural MEV thresholds")

pos = wallet_mev.filter(pl.col("n_arb_tx") > 0)
print(f"Wallets executing at least one cycle {pos.height:,} representing {pos.height/wallet_mev.height:.2%}")
print(f"Aggregate cyclic transactions {int(wallet_mev['n_arb_tx'].sum()):,}")

print("Quantile distribution of cyclic behavior")
for q in [0.5,0.75,0.9,0.95,0.99,0.999]:
    print(f"  Percentile {q*100:>5.1f} requires {pos['n_arb_tx'].quantile(q):>10,.0f} cyclic trades")

print("Concentration of total cyclic volume held by top tier algorithms")
tot = wallet_mev["n_arb_tx"].sum()
srt = wallet_mev.sort("n_arb_tx", descending=True)["n_arb_tx"]
for k in [100,1_000,10_000,100_000]:
    print(f"  Top {k:>7,} algorithms control {srt.head(k).sum()/tot:.1%} of cyclic activity")

# Establish nested sensitivity thresholds for econometric robustness testing
wallet_mev = wallet_mev.with_columns([
    (((pl.col("arb_tx_rate") > 0.50) & (pl.col("n_arb_tx") >= 5)) |
     (pl.col("n_arb_tx") >= 50) | (pl.col("n_sandwich_slots") >= 10)
    ).alias("mev_strict"),
    
    (((pl.col("arb_tx_rate") > 0.10) & (pl.col("n_arb_tx") >= 3)) |
     (pl.col("n_arb_tx") >= 10) | (pl.col("n_sandwich_slots") >= 3) |
     (pl.col("max_legs_one_tx") >= 6)
    ).alias("mev_base"),
    
    ((pl.col("n_arb_tx") >= 1) | (pl.col("n_sandwich_slots") >= 1)).alias("mev_loose"),
])

wallet_mev = wallet_mev.with_columns(pl.col("mev_base").alias("mev_wallet"))

for c in ["mev_strict","mev_base","mev_loose"]:
    k = int(wallet_mev[c].sum())
    print(f"Classification {c:<11} captured {k:>9,} wallets")

wallet_mev.write_parquet(OUT / "_H_wallet_mev.parquet", compression="zstd")

Calibrating structural MEV thresholds
Wallets executing at least one cycle 13,771 representing 0.35%
Aggregate cyclic transactions 1,009,597
Quantile distribution of cyclic behavior
  Percentile  50.0 requires          1 cyclic trades
  Percentile  75.0 requires          4 cyclic trades
  Percentile  90.0 requires         35 cyclic trades
  Percentile  95.0 requires        135 cyclic trades
  Percentile  99.0 requires        979 cyclic trades
  Percentile  99.9 requires      8,580 cyclic trades
Concentration of total cyclic volume held by top tier algorithms
  Top     100 algorithms control 68.5% of cyclic activity
  Top   1,000 algorithms control 94.3% of cyclic activity
  Top  10,000 algorithms control 99.6% of cyclic activity
  Top 100,000 algorithms control 100.0% of cyclic activity
Classification mev_strict  captured     2,130 wallets
Classification mev_base    captured    27,375 wallets
Classification mev_loose   captured    20,948 wallets


---
## Threshold Verification and Export
This cell reinforces the classifications generated in the previous step and runs boolean assertions to guarantee that the nested logic holds perfectly true (e.g., a strict MEV wallet must also mathematically qualify as a base MEV wallet). It then persists the validated data to disk.

In [25]:
wallet_mev = wallet_mev.with_columns(
    ((pl.col("n_arb_tx") >= 1) | (pl.col("n_sandwich_slots") >= 1) |
     (pl.col("max_legs_one_tx") >= 6)).alias("mev_loose")
)

# Assert mutual inclusivity to validate the nested heuristic logic
assert (wallet_mev["mev_strict"] & ~wallet_mev["mev_base"]).sum() == 0
assert (wallet_mev["mev_base"]   & ~wallet_mev["mev_loose"]).sum() == 0

for c in ["mev_strict","mev_base","mev_loose"]:
    k = int(wallet_mev[c].sum())
    print(f"Verified classification {c:<11} representing {k:>9,} wallets")

wallet_mev.write_parquet(OUT / "_H_wallet_mev.parquet", compression="zstd")

Verified classification mev_strict  representing     2,130 wallets
Verified classification mev_base    representing    27,375 wallets
Verified classification mev_loose   representing    41,328 wallets


---
## Bipartite Edge Generation and Wallet Topology
Calculating metrics across every unique wallet to pool connection requires immense RAM. This cell safely manages memory by partitioning the wallets into distinct hash buckets. Within each bucket, it calculates edge weights and topological concentration metrics like Shannon Entropy and Herfindahl Hirschman Index. High entropy signifies an exploratory or highly diversified algorithmic routing behavior, whereas high HHI indicates deep retail specialization in a single asset.

In [28]:
print("Constructing Bipartite Wallet to Pool Network Edges via Memory Bucketing")

NB = 16                      
PARTS = OUT / "_H_edge_parts"
if PARTS.exists(): 
    shutil.rmtree(PARTS)
PARTS.mkdir(parents=True)

# Define topological mathematical primitives for node profiling
def entropy(p): 
    return (pl.when(p > 0).then(-p * p.log()).otherwise(0.0)).sum()
def hhi(p):     
    return (p ** 2).sum()

# Distribute wallets deterministically across buckets
bkt = lambda c: pl.col(c).hash(seed=42).mod(NB)

W_parts = []
for b in range(NB):
    lf_b = (pl.scan_parquet(A)
        .filter(bkt(src["wallet"]) == b)
        .select([
            pl.col(src["wallet"]).alias("wallet"),
            pl.col(src["pool"]).alias("pool"),
            pl.col("tx_hash"),
            pl.col(src["ts"]).alias("ts"),
            pl.col(src["volume_usd"]).cast(pl.Float64).fill_null(0.0).abs().alias("volume_usd"),
        ]))

    arb_b = (pl.scan_parquet(BLK_CACHE)
               .filter(pl.col("is_cycle") & (bkt("wallet") == b))
               .select("tx_hash").unique()
               .with_columns(pl.lit(True).alias("arb_leg")))

    # Construct the localized network edges for the current bucket
    e = (lf_b.join(arb_b, on="tx_hash", how="left")
          .with_columns([
              (pl.col("ts").dt.year()*12 + pl.col("ts").dt.month()).cast(pl.Int32).alias("m"),
              pl.col("ts").dt.date().alias("day"),
              pl.col("arb_leg").fill_null(False),
          ])
          .group_by(["wallet","pool"])
          .agg([
              pl.len().cast(pl.UInt32).alias("n_swaps"),
              pl.col("tx_hash").n_unique().cast(pl.UInt32).alias("n_tx"),
              pl.col("volume_usd").sum().alias("volume_usd"),
              pl.col("ts").min().alias("_t0"), pl.col("ts").max().alias("_t1"),
              pl.col("m").min().alias("_m0"), pl.col("m").max().alias("_m1"),
              pl.col("m").n_unique().cast(pl.UInt16).alias("active_months"),
              pl.col("day").n_unique().cast(pl.UInt16).alias("active_days"),
              pl.col("arb_leg").mean().cast(pl.Float32).alias("arb_leg_share"),
          ])
          .with_columns([
              (pl.col("_m1")-pl.col("_m0")+1).cast(pl.UInt16).alias("span_months"),
              (pl.col("_t1")-pl.col("_t0")).dt.total_days().cast(pl.UInt16).alias("tenure_days"),
          ])
          .with_columns([
              (pl.col("active_months")/pl.col("span_months")).cast(pl.Float32).alias("edge_persistence"),
              (pl.col("span_months") >= 2).alias("edge_returned"),
              (pl.col("volume_usd")/pl.col("volume_usd").sum().over("wallet")).fill_nan(0.0)
                .cast(pl.Float32).alias("pw_vol"),
              (pl.col("n_swaps")/pl.col("n_swaps").sum().over("wallet"))
                .cast(pl.Float32).alias("pw_cnt"),
              (pl.col("volume_usd")/pl.col("n_swaps")).cast(pl.Float32).alias("avg_trade_usd"),
          ])
          .drop(["_t0","_t1","_m0","_m1"])
          .collect(engine="streaming"))

    # Aggregate edge attributes up to the Wallet Node level calculating topological diversification
    W_parts.append(
        e.group_by("wallet").agg([
            pl.len().cast(pl.UInt32).alias("degree_pools"),
            pl.col("n_swaps").sum().alias("w_swaps"),
            pl.col("n_tx").sum().alias("w_tx"),
            pl.col("volume_usd").sum().alias("w_volume_usd"),
            hhi(pl.col("pw_vol")).alias("hhi_vol"),
            hhi(pl.col("pw_cnt")).alias("hhi_cnt"),
            entropy(pl.col("pw_vol")).alias("entropy_vol"),
            entropy(pl.col("pw_cnt")).alias("entropy_cnt"),
            pl.col("pw_vol").max().alias("top_pool_share"),
            (pl.col("edge_persistence")*pl.col("pw_vol")).sum().alias("persistence_vw"),
            pl.col("edge_persistence").mean().alias("persistence_mean"),
            pl.col("edge_returned").mean().alias("share_pools_revisited"),
            pl.col("tenure_days").max().alias("max_edge_tenure_days"),
            pl.col("active_months").sum().alias("wallet_pool_months"),
        ]))

    e.write_parquet(PARTS / f"edges_{b:02d}.parquet", compression="zstd")
    print(f"Bucket {b+1:>2} complete yielding {e.height:>10,} edges")
    del e
    gc.collect()

W = pl.concat(W_parts)
del W_parts
gc.collect()

# Finalize Wallet Node metrics adjusting entropy relative to maximum possible network degree
W = (W.with_columns(pl.col("degree_pools").cast(pl.Float64).log().alias("_lnK"))
      .with_columns([
         pl.when(pl.col("degree_pools")>1).then(pl.col("entropy_vol")/pl.col("_lnK"))
           .otherwise(0.0).alias("entropy_norm_vol"),
         pl.when(pl.col("degree_pools")>1).then(pl.col("entropy_cnt")/pl.col("_lnK"))
           .otherwise(0.0).alias("entropy_norm_cnt"),
         pl.col("entropy_vol").exp().alias("eff_pools_exp_H"),
         (1.0/pl.col("hhi_vol")).alias("eff_pools_inv_HHI"),
      ])
      .with_columns([(1.0-pl.col("entropy_norm_vol")).alias("specialization"),
                     (1.0-pl.col("entropy_norm_cnt")).alias("specialization_cnt")])
      .drop("_lnK")
      .join(pl.read_parquet(OUT/"_H_wallet_mev.parquet"), on="wallet", how="left")
      .with_columns([pl.col(c).fill_null(False) for c in
                     ["mev_wallet","mev_strict","mev_base","mev_loose"]]))

W.write_parquet(OUT / "Dataset_H_Network_Wallets.parquet", compression="zstd")
print(f"Wallets mapped {W.height:,} nodes total width {W.width}")

Constructing Bipartite Wallet to Pool Network Edges via Memory Bucketing
Bucket  1 complete yielding    668,342 edges
Bucket  2 complete yielding    673,880 edges
Bucket  3 complete yielding    660,666 edges
Bucket  4 complete yielding    663,589 edges
Bucket  5 complete yielding    659,211 edges
Bucket  6 complete yielding    658,660 edges
Bucket  7 complete yielding    657,586 edges
Bucket  8 complete yielding    671,019 edges
Bucket  9 complete yielding    654,291 edges
Bucket 10 complete yielding    652,416 edges
Bucket 11 complete yielding    662,053 edges
Bucket 12 complete yielding    660,648 edges
Bucket 13 complete yielding    657,066 edges
Bucket 14 complete yielding    662,408 edges
Bucket 15 complete yielding    653,652 edges
Bucket 16 complete yielding    657,239 edges
Wallets mapped 3,987,674 nodes total width 35


---
## Venue Node Topologies and Mev Exposure Index
This cell calculates the topological characteristics of the liquidity pools (the venue nodes). It aggregates the total volume and identifies the Herfindahl Hirschman Index of the wallets trading within each pool. Crucially, it ranks pools based on their exposure to MEV algorithmic activity, generating a composite mev_exposure_index to quantify how heavily a specific market is targeted by extraction bots.

In [30]:
print("Calculating Pool Side Network Topology and Exposure Indexes")
EPG = str(PARTS / "edges_*.parquet")
bkt = lambda c: pl.col(c).hash(seed=42).mod(NB)

# Execute pass one global pool denominators to calculate relative market shares
ptot = (pl.scan_parquet(EPG).group_by("pool")
          .agg([pl.col("volume_usd").sum().alias("_pv"),
                pl.col("n_swaps").cast(pl.Int64).sum().alias("_pc")])
          .collect(engine="streaming"))
print(f"Total venue pools identified {ptot.height:,}")

mev_lf = (pl.scan_parquet(OUT/"_H_wallet_mev.parquet")
            .select(["wallet","mev_wallet","mev_strict","mev_loose"]))

# Execute pass two partial bucket sums combining edges and MEV exposure rates
P_parts, T_parts = [], []
for b in range(NB):
    eb = (pl.scan_parquet(PARTS/f"edges_{b:02d}.parquet")
            .join(mev_lf.filter(bkt("wallet") == b), on="wallet", how="left")
            .with_columns([pl.col(c).fill_null(False) for c in
                           ["mev_wallet","mev_strict","mev_loose"]])
            .join(ptot.lazy(), on="pool", how="left")
            .with_columns([
                (pl.col("volume_usd")/pl.col("_pv")).fill_nan(0.0).alias("pp_vol"),
                (pl.col("n_swaps").cast(pl.Float64)/pl.col("_pc")).alias("pp_cnt"),
            ]))

    P_parts.append(
        eb.group_by("pool").agg([
            pl.len().alias("k"),
            pl.col("n_swaps").cast(pl.Int64).sum().alias("s_swaps"),
            pl.col("volume_usd").sum().alias("s_vol"),
            (pl.col("pp_vol")**2).sum().alias("s_hhi_v"),
            (pl.col("pp_cnt")**2).sum().alias("s_hhi_c"),
            (pl.when(pl.col("pp_vol")>0).then(-pl.col("pp_vol")*pl.col("pp_vol").log())
               .otherwise(0.0)).sum().alias("s_ent"),
            pl.col("pp_vol").max().alias("mx"),
            pl.col("edge_persistence").cast(pl.Float64).sum().alias("s_pers"),
            pl.col("edge_returned").sum().alias("s_ret"),
            (pl.col("volume_usd")*pl.col("mev_wallet")).sum().alias("s_mv"),
            (pl.col("n_swaps").cast(pl.Int64)*pl.col("mev_wallet")).sum().alias("s_ms"),
            (pl.col("volume_usd")*pl.col("mev_strict")).sum().alias("s_mv_str"),
            pl.col("mev_wallet").sum().alias("s_mevw"),
            (pl.col("arb_leg_share").cast(pl.Float64)*pl.col("pp_cnt")).sum().alias("s_arb"),
        ]).collect(engine="streaming"))

    # Identify the concentration held by the top ten wallets per pool
    T_parts.append(eb.group_by("pool").agg(pl.col("pp_vol").top_k(10).alias("v"))
                     .explode("v").collect(engine="streaming"))
    print(f"Bucket {b+1:>2} aggregation complete")

top10 = (pl.concat(T_parts).group_by("pool")
           .agg(pl.col("v").top_k(10).sum().alias("top10_wallet_share")))
del T_parts

# Finalize the venue node metrics establishing mathematical exposure indices
P = (pl.concat(P_parts).group_by("pool").agg([
        pl.col("k").sum().alias("degree_wallets"),
        pl.col("s_swaps").sum().alias("p_swaps"),
        pl.col("s_vol").sum().alias("p_volume_usd"),
        pl.col("s_hhi_v").sum().alias("hhi_wallets_vol"),
        pl.col("s_hhi_c").sum().alias("hhi_wallets_cnt"),
        pl.col("s_ent").sum().alias("entropy_wallets"),
        pl.col("mx").max().alias("top_wallet_share"),
        pl.col("s_pers").sum().alias("_sp"), pl.col("s_ret").sum().alias("_sr"),
        pl.col("s_mv").sum().alias("_mv"), pl.col("s_ms").sum().alias("_ms"),
        pl.col("s_mv_str").sum().alias("_mvs"),
        pl.col("s_mevw").sum().alias("_mw"),
        pl.col("s_arb").sum().alias("arb_leg_share"),
     ])
     .join(top10, on="pool", how="left")
     .with_columns([
        (pl.col("_sp")/pl.col("degree_wallets")).alias("avg_wallet_persistence"),
        (pl.col("_sr")/pl.col("degree_wallets")).alias("repeat_wallet_share"),
        (pl.col("_mw")/pl.col("degree_wallets")).alias("mev_wallet_share"),
        (pl.col("_mv")/pl.col("p_volume_usd")).fill_nan(0.0).alias("mev_volume_share"),
        (pl.col("_mvs")/pl.col("p_volume_usd")).fill_nan(0.0).alias("mev_volume_share_strict"),
        (pl.col("_ms")/pl.col("p_swaps")).alias("mev_swap_share"),
        pl.when(pl.col("degree_wallets")>1)
          .then(pl.col("entropy_wallets")/pl.col("degree_wallets").cast(pl.Float64).log())
          .otherwise(0.0).alias("entropy_norm_wallets"),
        (1.0/pl.col("hhi_wallets_vol")).alias("eff_wallets_inv_HHI"),
     ]).drop(["_sp","_sr","_mv","_ms","_mvs","_mw"]))

# Generate a unified composite score reflecting comprehensive algorithmic exposure
comp = ["mev_volume_share","mev_swap_share","arb_leg_share"]
P = (P.with_columns([(pl.col(c).rank("average")/pl.len()).alias(f"_r{i}")
                     for i,c in enumerate(comp)])
      .with_columns(pl.mean_horizontal([f"_r{i}" for i in range(len(comp))])
                      .alias("mev_exposure_index"))
      .drop([f"_r{i}" for i in range(len(comp))])
      .sort("p_volume_usd", descending=True))

P.write_parquet(OUT / "Dataset_H_Network_Pools.parquet", compression="zstd")
P.write_csv(OUT / "Dataset_H_Network_Pools.csv")
print(f"Pools network topology established {P.height:,} nodes total width {P.width}")

Calculating Pool Side Network Topology and Exposure Indexes
Total venue pools identified 38,692


C:\Users\Pouyan\AppData\Local\Temp\ipykernel_14540\2803852077.py:49: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("v").collect(engine="streaming"))


Bucket  1 aggregation complete
Bucket  2 aggregation complete
Bucket  3 aggregation complete
Bucket  4 aggregation complete
Bucket  5 aggregation complete
Bucket  6 aggregation complete
Bucket  7 aggregation complete
Bucket  8 aggregation complete
Bucket  9 aggregation complete
Bucket 10 aggregation complete
Bucket 11 aggregation complete
Bucket 12 aggregation complete
Bucket 13 aggregation complete
Bucket 14 aggregation complete
Bucket 15 aggregation complete
Bucket 16 aggregation complete
Pools network topology established 38,692 nodes total width 19


## Final Bipartite Graph Assembly and Export
This concluding cell assembles the final components of the bipartite graph. It writes the entity level wallet file and the venue level pool file to disk. Finally, it concatenates the bucketed edge files into a unified master ledger representing the weighted connections between every trader and every smart contract in the dataset.

In [34]:
OUT = Path("./Thesis_Output")

print("Exporting Final Bipartite Network Graph Datasets")

# Export Entity Node metrics
W = pl.read_parquet(OUT / "Dataset_H_Network_Wallets.parquet")
W.write_csv(OUT / "Dataset_H_Network_Wallets.csv")
print(f"Wallet entity nodes saved representing {W.height:,} distinct actors")

# Export Venue Node metrics
P = pl.read_parquet(OUT / "Dataset_H_Network_Pools.parquet")
P.write_csv(OUT / "Dataset_H_Network_Pools.csv")
print(f"Pool venue nodes saved representing {P.height:,} distinct markets")

# Export Weighted Bipartite Edges connecting Wallets to Pools
EPG = str(OUT / "_H_edge_parts" / "edges_*.parquet")
edges = pl.scan_parquet(EPG).collect(engine="streaming")

edges.write_parquet(OUT / "Dataset_H_Network_Edges.parquet", compression="zstd")
print(f"Master network edges compiled {edges.height:,} distinct connections saved in Parquet format")

print("Writing edges to universal CSV format this will require significant disk space")
edges.write_csv(OUT / "Dataset_H_Network_Edges.csv")
print("Universal CSV export successful")

print("Network Topology pipeline is fully complete and secured in the output directory")

Exporting Final Bipartite Network Graph Datasets
Wallet entity nodes saved representing 3,987,674 distinct actors
Pool venue nodes saved representing 38,692 distinct markets
Master network edges compiled 10,572,726 distinct connections saved in Parquet format
Writing edges to universal CSV format this will require significant disk space
Universal CSV export successful
Network Topology pipeline is fully complete and secured in the output directory


---
## Results and Data Integrity
The architecture successfully compiled Dataset H, consisting of three distinct relational matrices: Wallets (node attributes), Pools (venue attributes), and Edges (weighted bipartite connections). The hash-bucketing strategy completely prevented memory exhaustion, and strict logical assertions verified the mutual exclusivity of the MEV classifications. 